# V18 – IT-Sicherheit für Ingenieure: Prüfsummen mit hashlib – Vertiefung zum Training

**Wo passt das Notebook hin?** Erst die Python-Werkstatt im Labor, dann das Training zu Hause (Browser, ca. 70 Min). Dieses Notebook ist die **Vertiefung** und läuft am besten im **Codespace** oder in Jupyter. Es nutzt Python aus V01 bis V17: Funktionen, Listen, Dictionaries, Dateien, try/except, Bibliotheken, und neu `hashlib` mit Bytes, Prüfsummen und Dateien, die blockweise gelesen werden.

**Der rote Faden:** die Fertigungszelle **FZ-3** in Halle 2. Nach der Störung an Roboter R1 kommt ein Servicetechniker mit der Firmware 3.2.1 auf einem USB-Stick. Der Hersteller veröffentlicht die Prüfsumme der Datei auf seiner Download-Seite: Stimmt die Datei vom Stick? Die Dateien liegen auf der Kurs-Website, die erste Zelle lädt sie neben das Notebook. Du brauchst eine Internetverbindung.

**Der Arbeitsablauf bei jeder Aufgabe:** (1) Die Daten ansehen (`print`), bevor du sie auswertest. (2) Dateien mit Prüfsummen immer **binär** (`"rb"`) lesen und den Soll-Wert auf die erste Spalte kürzen. (3) Zwei Prüfsummen nur als Text (`hexdigest()`) vergleichen. (4) Die Prüfzelle laufen lassen. Die Prüfzellen sind keine Musterlösungen, sie zeigen nur, ob dein Ergebnis stimmt.

**Stufen:** Aufwärmen (1.x), Anwenden (2.x), Knobeln (3.x). Lösungen gibt es hier nicht. Tipps sind eingeklappt.

In [ ]:
# Werkzeug-Zelle: einfach einmal ausführen, nicht verändern.
import hashlib
import json
import zipfile

import requests


def pruefe(ist, soll, text):
    if ist == soll:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")


def pruefe_ja(bedingung, text):
    if bedingung:
        print("OK: " + text)
    else:
        print("FEHLER: " + text)


# Adresse der Dateien (liegen als statische Dateien auf der Kurs-Website)
URL = "https://nice-wave-051f81f03.3.azurestaticapps.net/assets/data/v18/"


def hole(name):
    """Lädt eine Datei von der Kurs-Website und speichert sie neben dem Notebook (binär)."""
    antwort = requests.get(URL + name)
    if antwort.status_code != 200:
        print("Nicht ladbar:", name, "Status", antwort.status_code)
        return
    with open(name, "wb") as f:
        f.write(antwort.content)
    print("geladen:", name, len(antwort.content), "Bytes")


for name in ["R1_fw_3.2.1.bin", "R1_fw_3.2.1.bin.sha256", "R1_fw_3.2.1_usb.bin", "nc-freigabe.zip"]:
    hole(name)
zipfile.ZipFile("nc-freigabe.zip").extractall(".")      # entpackt den Ordner freigabe/ (NC-Programme)

## Stufe 1 – Aufwärmen

### 1.1 Text und Bytes
Eine Prüfsumme wird immer über **Bytes** gebildet. Wandle den Text mit `.encode("utf-8")` in Bytes um und lege fest: `daten`, die Zahl der Zeichen `n_zeichen` und die Zahl der Bytes `n_bytes`. Der Text hat zwei Umlaut-Zeichen: Das ö und das ß brauchen in UTF-8 je zwei Bytes.

<details><summary>Tipp anzeigen</summary>

Tipp: `daten = text.encode("utf-8")`. Zeichen zählt `len(text)`, Bytes zählt `len(daten)`.
</details>

In [ ]:
text = "Größe 20 mm"

# TODO: Text in UTF-8-Bytes umwandeln, dann Zeichen und Bytes zählen
daten = None
n_zeichen = None
n_bytes = None
print(daten, n_zeichen, n_bytes)

In [ ]:
pruefe(daten, "Größe 20 mm".encode("utf-8"), "daten sind die UTF-8-Bytes des Textes")
pruefe(n_zeichen, 11, "Der Text hat 11 Zeichen")
pruefe(n_bytes, 13, "Die Bytes sind zwei mehr: ö und ß brauchen je zwei")

### 1.2 Die erste Prüfsumme
Berechne die SHA-256-Prüfsumme des Textes „Drehzahl 1450“ als **Hex-Text** (`hexdigest()`) und lege sie in `pruefsumme` ab. Schau dir an, wie lang sie ist.

<details><summary>Tipp anzeigen</summary>

Tipp: `hashlib.sha256(...)` braucht Bytes: `"Drehzahl 1450".encode("utf-8")`. Danach `.hexdigest()`.
</details>

In [ ]:
import hashlib

text = "Drehzahl 1450"

# TODO: SHA-256 des Textes als Hex-Text
pruefsumme = None
print(pruefsumme)

In [ ]:
pruefe(pruefsumme, "23f2ab28ad6cc6b55540c120ae4291b7cd7cfc9fe48c1f9e1965c1d094f54958", "Prüfsumme von Drehzahl 1450")
pruefe(len(pruefsumme) if isinstance(pruefsumme, str) else None, 64, "64 Hex-Zeichen = 256 Bit")

### 1.3 Die Firmware auf einmal prüfen
Lies die Original-Firmware **binär** komplett ein (`daten`), lege die Größe in `groesse` ab und berechne ihre Prüfsumme als Hex-Text (`ist`). Lies dann die Datei `R1_fw_3.2.1.bin.sha256` (der Wert des Herstellers) und lege **nur die Prüfsumme** (erste Spalte) in `soll` ab.

<details><summary>Tipp anzeigen</summary>

Tipp: `open("R1_fw_3.2.1.bin", "rb").read()` liefert alle Bytes. Für die Prüfsummen-Datei: `open(..., encoding="utf-8").read().split()[0]`.
</details>

In [ ]:
import hashlib

# TODO: Firmware binär lesen, Größe und Prüfsumme bestimmen
daten = None
groesse = None
ist = None

# TODO: Soll-Wert (nur die Prüfsumme) aus der .sha256-Datei
soll = None
print(groesse, ist, soll)

In [ ]:
pruefe(groesse, 65536, "Die Firmware ist 65 536 Bytes groß")
pruefe(ist, "f457673f9e3b31fd836b6c71f76ef55cff73a0887186335f62906bd71d00e831", "Prüfsumme der Original-Firmware")
pruefe(soll, "f457673f9e3b31fd836b6c71f76ef55cff73a0887186335f62906bd71d00e831", "Soll-Wert: nur die 64 Zeichen, ohne Dateinamen")
pruefe(ist == soll, True, "Die Original-Datei passt zum Wert des Herstellers")

## Stufe 2 – Anwenden

### 2.1 Blockweise lesen
Die Firmware war nur 64 KiB groß. Ein echtes Abbild kann Gigabytes haben. Schreibe `sha256_datei(pfad)`: Die Funktion liest die Datei in Blöcken von **4096 Bytes** (`while True`, `read`, bei leerem Block `break`, `update`) und gibt die Prüfsumme als Hex-Text zurück. Rechne zusätzlich nach: `anzahl_bloecke` soll sagen, aus wie vielen Blöcken die Original-Datei besteht (Größe geteilt durch Blockgröße).

<details><summary>Tipp anzeigen</summary>

Tipp: Das Gerüst: `h = hashlib.sha256()`, `with open(pfad, "rb") as datei:`, `while True:` mit `block = datei.read(4096)`, `if not block: break`, `h.update(block)`. Am Ende `return h.hexdigest()`. Die Blockzahl: `65536 // 4096` (Ganzzahl-Division).
</details>

In [ ]:
import hashlib


def sha256_datei(pfad):
    # TODO: Datei blockweise lesen (4096 Bytes) und die Prüfsumme als Hex-Text zurückgeben
    return None


anzahl_bloecke = None      # TODO: Wie viele Blöcke zu 4096 Bytes hat die Original-Firmware?
print(sha256_datei("R1_fw_3.2.1.bin"))
print(sha256_datei("R1_fw_3.2.1_usb.bin"))
print(anzahl_bloecke)

In [ ]:
pruefe(sha256_datei("R1_fw_3.2.1.bin"), "f457673f9e3b31fd836b6c71f76ef55cff73a0887186335f62906bd71d00e831", "Original-Firmware")
pruefe(sha256_datei("R1_fw_3.2.1_usb.bin"), "c01a88afebff47f13d689931bc16a24868e108c837db8c48f291beb0160ef54f", "Datei vom USB-Stick")
pruefe(anzahl_bloecke, 16, "16 Blöcke zu 4096 Bytes")
with open("t_klein.bin", "wb") as f:
    f.write(b"abc")
pruefe(sha256_datei("t_klein.bin"), "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad", "Auch eine Datei, die kleiner als ein Block ist")

### 2.2 Bits zählen
Im Labor hast du gesehen: Ein einziges gekipptes Bit in der Firmware verändert die Prüfsumme komplett. Miss es: Schreibe `bits_unterschied(h1, h2)`, die zwei Hex-Prüfsummen bekommt und zurückgibt, in wie vielen der 256 Bits sie sich unterscheiden. Wende sie auf die Original-Firmware und die Datei vom Stick an (`bits`) und berechne den Anteil `anteil` (Bits geteilt durch 256).

Neu: `^` vergleicht zwei Zahlen Bit für Bit (XOR, ein Bit im Ergebnis ist 1, wenn die Bits verschieden sind). `bin(...)` schreibt eine Zahl binär, `.count("1")` zählt die Einsen.

<details><summary>Tipp anzeigen</summary>

Tipp: `int(h1, 16)` macht aus Hex-Text eine Zahl. `bin(int(h1, 16) ^ int(h2, 16)).count("1")` ist die Zahl der verschiedenen Bits.
</details>

In [ ]:
# sha256_datei aus 2.1 steht hier schon, damit diese Aufgabe für sich läuft
import hashlib


def sha256_datei(pfad):
    h = hashlib.sha256()
    with open(pfad, "rb") as datei:
        while True:
            block = datei.read(4096)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def bits_unterschied(h1, h2):
    # TODO
    return None


original = sha256_datei("R1_fw_3.2.1.bin")
vom_stick = sha256_datei("R1_fw_3.2.1_usb.bin")
bits = bits_unterschied(original, vom_stick)
anteil = None              # TODO: Anteil der verschiedenen Bits an den 256 Bits
print(bits, anteil)

In [ ]:
pruefe(bits_unterschied("0f", "00"), 4, "0f gegen 00: vier Bits")
pruefe(bits_unterschied("a5", "5a"), 8, "a5 gegen 5a: alle acht Bits")
pruefe(bits, 121, "Firmware: so viele Bits der Prüfsumme sind verschieden")
pruefe_ja(anteil is not None and 0.35 < anteil < 0.65, "Etwa die Hälfte der Bits ist verschieden (Lawineneffekt)")

### 2.3 Dateien, die nicht in der Freigabeliste stehen
Im Labor hast du Dateien gegen die Freigabeliste geprüft. Es gibt noch einen zweiten Fall: Jemand legt eine **neue** Datei in den Ordner, die nie freigegeben wurde. Die erste Zelle legt so eine Datei an. Schreibe `nicht_freigegeben(ordner)`: Sie gibt eine **sortierte Liste** aller Pfade (relativ zum Ordner, mit `/`) der Dateien im Unterordner `nc-programme` zurück, die **nicht** in `freigabe.sha256` stehen.

Die Liste hat je Zeile `<Prüfsumme>  <Pfad>`. Mit `pathlib` (V16): `Path(ordner, "nc-programme").glob("*.nc")` findet Dateien, `datei.name` ist der Dateiname.

<details><summary>Tipp anzeigen</summary>

Tipp: Lies die Pfade aus der Liste in eine Menge oder Liste (zweite Spalte jeder Zeile: `zeile.split()[1]`). Dann für jede `.nc`-Datei den Pfad `"nc-programme/" + datei.name` bilden und prüfen, ob er in der Liste fehlt. `sorted(...)` sortiert.
</details>

In [ ]:
# Jemand hat eine neue Datei abgelegt, die nie freigegeben wurde
with open("freigabe/nc-programme/TEST_ALT.nc", "w", encoding="utf-8") as f:
    f.write("%\nO9999 (TEST)\nM30\n%\n")

In [ ]:
from pathlib import Path


def nicht_freigegeben(ordner):
    # TODO: Pfade aus freigabe.sha256 lesen, dann Dateien in nc-programme suchen,
    # die nicht in der Liste stehen (sortierte Liste der Pfade zurückgeben)
    return None


print(nicht_freigegeben("freigabe"))

In [ ]:
pruefe(nicht_freigegeben("freigabe"), ["nc-programme/TEST_ALT.nc"], "Nur TEST_ALT.nc steht nicht in der Freigabeliste")
import os
os.makedirs("t_frei/nc-programme", exist_ok=True)
for name in ["b.nc", "a.nc", "c.nc"]:
    with open("t_frei/nc-programme/" + name, "w", encoding="utf-8") as f:
        f.write("M30\n")
with open("t_frei/freigabe.sha256", "w", encoding="utf-8") as f:
    f.write("00  nc-programme/b.nc\n\n")
pruefe(nicht_freigegeben("t_frei"), ["nc-programme/a.nc", "nc-programme/c.nc"], "Eigener Ordner: sortiert, Leerzeilen stören nicht")

## Stufe 3 – Knobeln

### 3.1 Einwegfunktion: nur probieren hilft
Ein Hash ist eine **Einwegfunktion**: Aus dem Hash kommst du nicht auf die Daten zurück, du kannst nur probieren. Finde die kleinste Zahl `n` (0, 1, 2, ...), für die der SHA-256-Hash von `text + str(n)` mit einer bestimmten Zahl von Nullen beginnt (Hex-Nullen, `startswith`). Schreibe `finde_nonce(text, nullen)`, die diese kleinste Zahl zurückgibt.

Frage zum Nachdenken: Jede weitere Hex-Null macht die Suche im Schnitt 16-mal länger. Warum? (Ein Hex-Zeichen hat 16 mögliche Werte.)

<details><summary>Tipp anzeigen</summary>

Tipp: Eine Schleife mit Zähler `n = 0`: Hash von `(text + str(n)).encode("utf-8")` bilden, mit `startswith("0" * nullen)` prüfen, sonst `n = n + 1`. Bei vier Nullen sind es im Schnitt 65 536 Versuche, das geht in wenigen Sekunden.
</details>

In [ ]:
import hashlib


def finde_nonce(text, nullen):
    # TODO: kleinste Zahl n, sodass sha256(text + str(n)) mit `nullen` Hex-Nullen beginnt
    return None


n3 = finde_nonce("R1-FW 3.2.1 ", 3)
n4 = finde_nonce("R1-FW 3.2.1 ", 4)
print(n3, n4)

In [ ]:
pruefe(n3, 8587, "Kleinste Zahl für drei Nullen")
pruefe(n4, 19836, "Kleinste Zahl für vier Nullen")
pruefe(finde_nonce("x", 0), 0, "Ohne Nullen genügt schon die 0")

### 3.2 Hash-Kette
Ein Protokoll, das nachträglich geändert werden kann, ist wenig wert. Mit einer **Hash-Kette** geht jede Zeile samt dem Hash der vorigen in den neuen Hash ein: `hash_i = sha256(hash_(i-1) + zeile_i)`, der Anfangswert ist `"0" * 64`. Schreibe `baue_protokoll(zeilen)`, die eine Liste von Paaren `(zeile, hash)` zurückgibt, und `erste_manipulation(protokoll)`, die den Index der ersten Zeile liefert, deren gespeicherter Hash nicht zur Kette passt (sonst `None`).

<details><summary>Tipp anzeigen</summary>

Tipp: Beide Funktionen laufen mit demselben Muster: `vorher = "0" * 64`, für jede Zeile `neu = sha256((vorher + zeile).encode("utf-8")).hexdigest()`. Beim Bauen speicherst du `(zeile, neu)`. Beim Prüfen vergleichst du `neu` mit dem gespeicherten Hash und gibst bei Abweichung den Index zurück. Danach ist der gespeicherte Hash das neue `vorher`.
</details>

In [ ]:
import hashlib


def baue_protokoll(zeilen):
    # TODO: Liste von (zeile, hash) mit verketteten Hashes
    return None


def erste_manipulation(protokoll):
    # TODO: Index der ersten Zeile, deren Hash nicht zur Kette passt, sonst None
    return None


zeilen = ["08:02 Greifer zu", "08:03 Teil abgelegt", "08:05 Lichtgitter unterbrochen", "08:05 Not-Halt"]
protokoll = baue_protokoll(zeilen)
print(erste_manipulation(protokoll))

In [ ]:
pruefe(len(protokoll), 4, "Vier Einträge")
pruefe(protokoll[0][1], hashlib.sha256(("0" * 64 + "08:02 Greifer zu").encode("utf-8")).hexdigest(), "Erster Hash: Anfangswert plus erste Zeile")
pruefe(protokoll[1][1], hashlib.sha256((protokoll[0][1] + "08:03 Teil abgelegt").encode("utf-8")).hexdigest(), "Zweiter Hash baut auf dem ersten auf")
pruefe(erste_manipulation(protokoll), None, "Unverändertes Protokoll ist in Ordnung")
geaendert = list(protokoll)
geaendert[2] = ("08:05 Lichtgitter frei", geaendert[2][1])
pruefe(erste_manipulation(geaendert), 2, "Geänderte Zeile 2 wird gefunden")
geloescht = list(protokoll)
del geloescht[1]
pruefe(erste_manipulation(geloescht), 1, "Gelöschte Zeile: die Kette reißt an der Lücke")

### 3.3 Eigene Frage an die Daten (frei)
Wähle selbst: Wie schnell kann dein Rechner Hashes berechnen? Miss mit `time` (V16) die Zeit für 100 000 SHA-256-Hashes kurzer Texte, und rechne hoch, wie lange ein Angreifer für alle 4-stelligen PINs (10 000) und für alle 8-stelligen Passwörter aus Ziffern (100 Millionen) bräuchte. Was folgt daraus für das Speichern von Passwörtern? (Die Lerneinheit nennt gesalzene, absichtlich langsame Verfahren.) Es gibt keine Prüfzelle. Erkläre dein Ergebnis einem Mitstudierenden.

In [ ]:
# Dein Code


## Ausblick: Signaturen (freiwillig)
Eine Prüfsumme sagt: „Die Datei passt zu diesem Wert.“ Sie sagt nicht: „Der Hersteller hat diese Datei freigegeben.“ Dafür gibt es die **digitale Signatur**: Der Hersteller verschlüsselt den Hash der Firmware mit seinem **privaten Schlüssel**, das Gerät prüft mit dem **öffentlichen Schlüssel**, ob die Signatur zum Hash der Datei passt. Nur der Hersteller kann so signieren. Mit **Secure Boot** startet die Steuerung nur Firmware mit gültiger Signatur. Die Mathematik (RSA) steht im Deep Dive der Lerneinheit; in der Praxis nutzt du fertige Werkzeuge der Hersteller.

## Selbsttest: Kannst du es im Kopf?
Beantworte erst auf Papier, dann prüfe im Notebook.

1. Was gibt `len("Ä".encode("utf-8"))` aus, und wie viele Hex-Zeichen hat eine SHA-256-Prüfsumme?
2. Warum beweist eine übereinstimmende Prüfsumme nichts, wenn Firmware und Prüfsumme vom selben USB-Stick kommen? Woher muss der Soll-Wert kommen?
3. Was ist die Prüfsumme (die ersten 8 Zeichen) einer leeren Datei?
4. Du änderst in einer Datei ein einziges Bit. Wie viele Bits der Prüfsumme ändern sich etwa?

In [ ]:
import hashlib

print(len("Ä".encode("utf-8")), len(hashlib.sha256(b"").hexdigest()))     # 1: 2 und 64
print("Wer die Datei tauscht, tauscht auch die Prüfsumme. Der Soll-Wert braucht einen zweiten, vertrauenswürdigen Weg (Hersteller-Website per HTTPS) - besser: Signatur.")   # 2
print(hashlib.sha256(b"").hexdigest()[:8])                                # 3: e3b0c442
print("etwa die Hälfte, also rund 128 von 256 Bits (Lawineneffekt)")     # 4